# XGB One-Versus-Rest — Improved
Key improvements over v001:
- **Original data used for training** (was loaded but never trained on)
- **All FE applied to `orig`** (digit, threshold, logit features)
- **StratifiedKFold** instead of plain KFold
- **Ratio / interaction features** between main numeric drivers
- **3-way categorical combos** enabled
- **500 Optuna trials** (was 200) for class-weight post-processing
- XGB: added `gamma`, lower `learning_rate` for precision


In [ ]:
NAME = '002'

In [ ]:
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
import seaborn as sns

In [ ]:
%load_ext cudf.pandas

import numpy as np, pandas as pd, gc
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 200)

In [ ]:
train_file = '/kaggle/input/competitions/playground-series-s6e4/train.csv'
test_file  = '/kaggle/input/competitions/playground-series-s6e4/test.csv'
original_file = '/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv'

train = pd.read_csv(train_file)
test  = pd.read_csv(test_file)
orig  = pd.read_csv(original_file)

train_ids = train['id'].copy()
test_ids  = test['id'].copy()  # for submission

print(f"train: {train.shape}, test: {test.shape}, orig: {orig.shape}")

In [ ]:
train['Irrigation_Need'] = train['Irrigation_Need'].map(_classes := {'Low': 0, 'Medium': 1, 'High': 2})
orig['Irrigation_Need']  = orig['Irrigation_Need'].map(_classes)

In [ ]:
TARGET = 'Irrigation_Need'
NUMS = list(train.drop(['id', TARGET], axis=1)._get_numeric_data().columns)
CATS = [c for c in train.drop(['id', TARGET], axis=1).columns if c not in NUMS]

# simple check: id + target = 2
assert len(NUMS) + len(CATS) + 2 == train.shape[1]

print(f"Categorical columns ({len(CATS)}): {CATS}")
print(f"Numerical  columns  ({len(NUMS)}): {NUMS}")

# Feature Engineering

In [ ]:
NEW_NUMS   = []
NEW_CATS   = []
NUM_AS_CAT = []
TO_REMOVE  = []
NON_TE_CATS = []

In [ ]:
_to_combo = CATS

for i, c1 in enumerate(_to_combo[:-1]):
    for j, c2 in enumerate(_to_combo[i + 1:]):
        _new_col = f'COMBO_{c1}_{c2}'
        for df in [train, test, orig]:
            df[_new_col] = df[c1].astype('str') + '_' + df[c2].astype('str')
        NEW_CATS.append(_new_col)

        # 3-way combos (enabled — adds useful signal)
        for k, c3 in enumerate(_to_combo[i+j+2:]):
            _new_col3 = f'COMBO_{c1}_{c2}_{c3}'
            for df in [train, test, orig]:
                df[_new_col3] = df[c1].astype('str') + '_' + df[c2].astype('str') + '_' + df[c3].astype('str')
            NEW_CATS.append(_new_col3)

print(f"Total NEW_CATS so far (combos): {len(NEW_CATS)}")

In [ ]:
# Compute frequencies from ALL data (train + orig + test)
for cat in CATS + NEW_CATS:
    freq = pd.concat([train[cat], orig[cat], test[cat]]).value_counts(normalize=True)
    for df in [train, test, orig]:
        df[f'FREQ_{cat}'] = df[cat].map(freq).fillna(0).astype('float32')
    NEW_NUMS.append(f'FREQ_{cat}')

print(f"Freq features added: {len([c for c in NEW_NUMS if c.startswith('FREQ_')])}")

In [ ]:
# numerical as categorical
for col in NUMS:
    _new_col = f'CAT_{col}'
    NUM_AS_CAT.append(_new_col)
    for df in [train, test, orig]:
        df[_new_col] = df[col].astype(str).astype('category')

In [ ]:
# adapted from https://www.kaggle.com/code/yunsuxiaozi/pss6e4-xgb-cv-0-979805
# FIX: now applied to orig as well so orig can be used in training

M = train[NUMS].max()
DIGIT_FEATURES = []

for c in NUMS:
    for df in [train, test, orig]:   # <-- orig added
        for k in range(-4, 4):
            df[f"{c}_digit{k}"] = (df[c] // (10**k) % 10).astype('int8')
            DIGIT_FEATURES += [f"{c}_digit{k}"]

    # Rounding (use max from train for consistency)
    for df in [train, test, orig]:
        if M[c] < 10:
            df[c] = df[c].round(3)
        elif M[c] < 100:
            df[c] = df[c].round(2)
        else:
            df[c] = df[c].round(1)

DIGIT_FEATURES = list(set(DIGIT_FEATURES))

DROP = [c for c in test.columns if test[c].nunique() == 1]
print(f"DROP: {DROP}")

train.drop(DROP, axis=1, inplace=True)
test.drop(DROP,  axis=1, inplace=True)
orig.drop([c for c in DROP if c in orig.columns], axis=1, inplace=True)

DIGIT_FEATURES = list(set(DIGIT_FEATURES) - set(DROP))
NEW_CATS += DIGIT_FEATURES

see https://www.kaggle.com/code/cdeotte/original-data-exact-formula for exact formula for `orig`

In [ ]:
# FIX: threshold features now applied to orig
for df in [train, test, orig]:   # <-- orig added
    df["soil_lt_25"]   = (df["Soil_Moisture"]   < 25).astype(int)
    df["temp_gt_30"]   = (df["Temperature_C"]   > 30).astype(int)
    df["rain_lt_300"]  = (df["Rainfall_mm"]      < 300).astype(int)
    df["wind_gt_10"]   = (df["Wind_Speed_kmh"]   > 10).astype(int)

TRES_CATS = ['soil_lt_25', 'temp_gt_30', 'rain_lt_300', 'wind_gt_10']
NEW_CATS += TRES_CATS

In [ ]:
# FIX: logit features now applied to orig as well
for df_ in [train, test, orig]:   # <-- orig added
    df = pd.get_dummies(
        df_[NUMS + CATS + TRES_CATS],
        columns=CATS,
        drop_first=False
    )
    df_['logit(P(y=Low))']    = ( 16.3173 + (-11.0237 * df["soil_lt_25"]) + (-5.8559 * df["temp_gt_30"])
        + (-10.8500 * df["rain_lt_300"]) + (-5.8284 * df["wind_gt_10"])
        + (-5.4155 * df["Crop_Growth_Stage_Flowering"]) + (5.5073 * df["Crop_Growth_Stage_Harvest"])
        + (5.2299  * df["Crop_Growth_Stage_Sowing"])   + (-5.4617 * df["Crop_Growth_Stage_Vegetative"])
        + (-3.0014 * df["Mulching_Used_No"])            + (2.8613 * df["Mulching_Used_Yes"]))
    df_['logit(P(y=Medium))'] = ( 4.6524 + (0.3290 * df["soil_lt_25"]) + (-0.0204 * df["temp_gt_30"])
        + (0.1542 * df["rain_lt_300"]) + (0.0841 * df["wind_gt_10"])
        + (0.3586 * df["Crop_Growth_Stage_Flowering"]) + (-0.1348 * df["Crop_Growth_Stage_Harvest"])
        + (-0.3547 * df["Crop_Growth_Stage_Sowing"])   + (0.3334  * df["Crop_Growth_Stage_Vegetative"])
        + (0.1883 * df["Mulching_Used_No"])             + (0.0142 * df["Mulching_Used_Yes"]))
    df_['logit(P(y=High))']   = (-20.9697 + (10.6947 * df["soil_lt_25"]) + (5.8763 * df["temp_gt_30"])
        + (10.6958 * df["rain_lt_300"]) + (5.7444 * df["wind_gt_10"])
        + (5.0569  * df["Crop_Growth_Stage_Flowering"]) + (-5.3725 * df["Crop_Growth_Stage_Harvest"])
        + (-4.8752 * df["Crop_Growth_Stage_Sowing"])    + (5.1283  * df["Crop_Growth_Stage_Vegetative"])
        + (2.8131  * df["Mulching_Used_No"])             + (-2.8755 * df["Mulching_Used_Yes"]))

NEW_NUMS += ['logit(P(y=Low))', 'logit(P(y=Medium))', 'logit(P(y=High))']

In [ ]:
train.isna().any().any(), test.isna().any().any(), orig.isna().any().any()

In [ ]:
# FIX: also merge ORIG stats into orig itself so it can be used in training
for col in CATS + NUMS:
    stats = orig.groupby(col)[TARGET].agg(['mean', 'std']).reset_index()
    stats.columns = [col] + [f"ORIG_{col}_{s}" for s in ['mean', 'std']]
    fill_values = {f"ORIG_{col}_mean": 0.5, f"ORIG_{col}_std": 0}

    for df in [train, test, orig]:   # <-- orig added
        df.merge(stats, on=col, how='left').pipe(lambda d: d.fillna(value=fill_values))

    train = train.merge(stats, on=col, how='left').fillna(value=fill_values)
    test  = test.merge(stats, on=col, how='left').fillna(value=fill_values)
    orig  = orig.merge(stats, on=col, how='left').fillna(value=fill_values)

    NEW_NUMS.extend([f"ORIG_{col}_{s}" for s in ['mean', 'std']])

In [ ]:
# NEW: ratio and interaction features between the main numeric drivers.
# The logit formula tells us these 4 numerics are the key drivers.
# Their ratios capture relative relationships the logit features can't.

EPS = 1e-6
for df in [train, test, orig]:
    # Water availability composite
    df['water_avail']     = df['Soil_Moisture'] + df['Rainfall_mm'] / 300.0
    # Heat-to-moisture stress ratio
    df['heat_stress']     = df['Temperature_C'] / (df['Soil_Moisture'] + EPS)
    # Rain cooling relative to temperature
    df['rain_cool']       = df['Rainfall_mm'] / (df['Temperature_C'] + EPS)
    # Wind evaporation stress
    df['wind_evap']       = df['Wind_Speed_kmh'] * df['Temperature_C'] / 100.0
    # Soil-rain product (both must be present for Low irrigation)
    df['soil_x_rain']     = df['Soil_Moisture'] * df['Rainfall_mm'] / 1000.0
    # High temp + low moisture -> High irrigation signal
    df['temp_x_dryness']  = df['Temperature_C'] * (100 - df['Soil_Moisture']) / 100.0

RATIO_FEATURES = ['water_avail', 'heat_stress', 'rain_cool', 'wind_evap', 'soil_x_rain', 'temp_x_dryness']
NEW_NUMS += RATIO_FEATURES
print(f"Ratio features added: {RATIO_FEATURES}")

In [ ]:
FEATURES = NUMS + CATS + NEW_NUMS + NEW_CATS + NUM_AS_CAT + NON_TE_CATS
print(f'We now have {len(FEATURES)} columns:')
print(FEATURES)

In [ ]:
TE_COLUMNS = NUM_AS_CAT + CATS + NEW_CATS
TO_REMOVE += NUM_AS_CAT + CATS + NEW_CATS
QUANTILE_COLUMNS = []

# Model Training

In [ ]:
np.random.seed(11)

In [ ]:
FOLDS       = 5
INNER_FOLDS = 5

In [ ]:
xgb_params = {
    'n_estimators':    50000,
    'max_depth':       4,
    'colsample_bytree': 0.8,
    'subsample':       0.8,
    'learning_rate':   0.05,     # lowered from 0.1 for more precision
    'n_jobs':          -1,
    'enable_categorical': True,
    'alpha':           5,
    'reg_lambda':      5,
    'gamma':           0.1,      # NEW: min split loss regularisation
    'max_leaves':      30,
    'min_child_weight': 2,
    'tree_method':     'hist',
    'max_bin':         10000,
    'device':          'cuda',
    'objective':       'binary:logistic',  # binary for OVR
}

# Ordered Target Encoder

In [ ]:
from functools import reduce

class OrderedTE:
    def __init__(self, a=1):
        self.a = a

    def fit(self, train, category_cols=(), target_col='target'):
        self.category_cols = category_cols
        self.classes_ = sorted(train[target_col].unique())
        self.global_prior_ = train[target_col].value_counts(normalize=True).sort_index().values
        self.stats_ = {}

        for c in category_cols:
            stats_list = []
            for k, cls in enumerate(self.classes_):
                y = (train[target_col] == cls).astype(int)
                grp = train[[c]].assign(y=y.values)
                cum_cnt = grp.groupby(c, observed=False)['y'].cumcount()
                cum_sum = grp.groupby(c, observed=False)['y'].cumsum() - grp['y']
                prior = self.global_prior_[k]
                te = (cum_sum + self.a * prior) / (cum_cnt + self.a)
                te_col = f'{c}_TE_cls{cls}'
                train[te_col] = te.values
                agg = grp.groupby(c, observed=False)['y'].agg(count='count', total='sum').reset_index()
                agg.columns = [c, f'{c}_n_{cls}', f'{c}_s_{cls}']
                stats_list.append(agg)
            self.stats_[c] = reduce(lambda l, r: l.merge(r, on=c, how='outer'), stats_list)
        return train

    def transform(self, test):
        for c in self.category_cols:
            test = test.merge(self.stats_[c], on=c, how='left')
            for k, cls in enumerate(self.classes_):
                te_col = f'{c}_TE_cls{cls}'
                n_col, s_col = f'{c}_n_{cls}', f'{c}_s_{cls}'
                prior = self.global_prior_[k]
                if n_col in test.columns:
                    test[te_col] = ((test[s_col] + self.a * prior) / (test[n_col] + self.a)).fillna(prior)
                    test.drop(columns=[n_col, s_col], inplace=True)
                else:
                    test[te_col] = prior
        return test

# Model Training — StratifiedKFold + Original Data

In [ ]:
def metric(y_true, y_pred, sample_weight=None):
    y_pred = np.argmax(y_pred, axis=1)
    return balanced_accuracy_score(y_true, y_pred, sample_weight=sample_weight)

In [ ]:
# %%time

print(f"\n{'='*80}")
print("TRAINING XGBOOST ONE-VERSUS-REST  (with original data in training)")
print("="*80)

# FIX: StratifiedKFold for class-balanced splits
skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=11)

# Prepare orig feature columns (same as train, minus id, keep TARGET for TE fit)
orig_train = orig[FEATURES + [TARGET]].reset_index(drop=True).copy()

oof         = np.zeros((len(train), 3))
pred        = np.zeros((len(test),  3))
importances = {0: [], 1: [], 2: []}
metric_folds = []

for i, (train_index, val_index) in enumerate(skf.split(train, train[TARGET])):
    print(f"\n{'='*60}")
    print(f"Fold {i+1}/{FOLDS}")
    print('='*60)

    # ── TRAIN/VAL SPLIT ──────────────────────────────────────────
    X_train = train.loc[train_index, FEATURES + [TARGET]].reset_index(drop=True).copy()
    y_train_multiclass = train.loc[train_index, TARGET].values

    X_val = train.loc[val_index, FEATURES].reset_index(drop=True).copy()
    y_val_multiclass = train.loc[val_index, TARGET].values

    # FIX: include orig in training data
    # We concat BEFORE TE so the TE is fitted on train-fold only,
    # then transform orig separately and concatenate for actual model fit.
    X_orig_for_te = orig_train.copy()  # has TARGET for TE fit reference
    X_test = test[FEATURES].reset_index(drop=True).copy()

    # ── TARGET ENCODING ──────────────────────────────────────────
    target_encoder = OrderedTE(a=1)
    X_train = target_encoder.fit(X_train, category_cols=TE_COLUMNS, target_col=TARGET)
    X_val   = target_encoder.transform(X_val)
    X_test  = target_encoder.transform(X_test)

    # Transform orig with the same TE (fitted on train fold only)
    X_orig_features = X_orig_for_te.drop(TARGET, axis=1)
    X_orig_features = target_encoder.transform(X_orig_features)
    y_orig_multiclass = X_orig_for_te[TARGET].values

    # ── CONVERT CATS FOR XGBOOST ─────────────────────────────────
    for df in [X_train, X_val, X_test, X_orig_features]:
        df[CATS + NEW_CATS + NUM_AS_CAT + NON_TE_CATS] = (
            df[CATS + NEW_CATS + NUM_AS_CAT + NON_TE_CATS]
            .astype(str).astype("category")
        )

    # Drop raw cat columns (now replaced by TE)
    for df in [X_train, X_val, X_test, X_orig_features]:
        drop_cols = [c for c in TO_REMOVE if c in df.columns]
        df.drop(drop_cols, axis=1, inplace=True)

    X_train = X_train.drop([TARGET], axis=1)
    COLS = X_train.columns

    # ── CONCAT train-fold + orig for model training ───────────────
    X_train_full = pd.concat([X_train, X_orig_features[COLS]], axis=0).reset_index(drop=True)
    y_train_full  = np.concatenate([y_train_multiclass, y_orig_multiclass])

    # ── TRAIN 3 BINARY OVR MODELS ────────────────────────────────
    fold_oof_probs  = np.zeros((len(X_val),    3))
    fold_test_probs = np.zeros((len(X_test),   3))

    for class_idx in range(3):
        print(f"\n  Training binary classifier for class {class_idx}")

        y_train_binary = (y_train_full == class_idx).astype(int)
        y_val_binary   = (y_val_multiclass == class_idx).astype(int)

        s_wei_train = compute_sample_weight('balanced', y_train_binary)

        model = xgb.XGBClassifier(
            **xgb_params,
            eval_metric='logloss',
            callbacks=[
                xgb.callback.EarlyStopping(
                    rounds=300, metric_name='logloss', save_best=True
                )
            ],
        )
        model.fit(
            X_train_full, y_train_binary,
            sample_weight=s_wei_train,
            eval_set=[(X_val, y_val_binary)],
            verbose=1000
        )

        fold_oof_probs[:, class_idx]  = model.predict_proba(X_val)[:, 1]
        fold_test_probs[:, class_idx] = model.predict_proba(X_test[COLS])[:, 1]

        importances[class_idx].append(model.get_booster().get_score(importance_type='gain'))

    # Normalise to sum-to-1
    fold_oof_probs  /= fold_oof_probs.sum(axis=1,  keepdims=True)
    fold_test_probs /= fold_test_probs.sum(axis=1,  keepdims=True)

    oof[val_index] = fold_oof_probs
    pred += fold_test_probs

    bal_acc_fold = metric(y_val_multiclass, fold_oof_probs)
    metric_folds.append(bal_acc_fold)
    print(f"\nFold {i+1} Validation Balanced Accuracy: {bal_acc_fold:.5f}")

    del X_train, X_val, X_test, X_train_full, X_orig_features
    del y_train_multiclass, y_val_multiclass, y_train_full, y_orig_multiclass
    gc.collect()

pred /= FOLDS

In [ ]:
print(f'Fold Balanced Accuracy {np.mean(metric_folds):.5f} +- {np.std(metric_folds):.5f}')
true = train[TARGET].values
print(f'Overall Balanced Accuracy: {metric(true, oof)}')

In [ ]:
print(f'\nIn total, we used {len(COLS)} features, Wow!\n')
print(list(COLS))

# XGB Feature Importance (per class)

In [ ]:
all_importances = []
for class_idx in range(3):
    for fold_imp in importances[class_idx]:
        all_importances.append(fold_imp)

feature_names = all_importances[0].keys()
importances_mean = [
    np.mean([imp[feat] if feat in imp else 0 for imp in all_importances])
    for feat in feature_names
]

In [ ]:
indices = np.argsort(importances_mean)
sorted_features   = np.array(list(feature_names))[indices][-100:]
sorted_importance = np.array(importances_mean)[indices][-100:]

plt.figure(figsize=(15, 30))
plt.barh(range(len(sorted_importance)), sorted_importance)
plt.yticks(range(len(sorted_features)), sorted_features)
plt.xlabel('Feature Importance')
plt.ylabel('Features')
plt.title('XGBoost Feature Importance (Sorted)')
plt.tight_layout()
plt.show()

In [ ]:
dict(sorted(zip(feature_names, [float(f) for f in importances_mean]), key=lambda x: -x[1]))

In [ ]:
def accuracy_score(t, p):
    if len(p.shape) == 2:
        p = np.argmax(p, axis=1)
    C = 3
    acc = 0.0
    for i in range(C):
        acc += np.sum((t == i) & (p == i)) / np.sum(t == i) / C
    return acc

# Post-processing: Optuna Class Weight Optimisation

In [ ]:
oof_preds = oof
y = train[TARGET]

import optuna
from optuna.samplers import TPESampler

def objective(trial):
    cw1 = trial.suggest_float('cw1', 0.5, 3.0)
    cw2 = trial.suggest_float('cw2', 0.5, 3.0)
    cw3 = trial.suggest_float('cw3', 0.5, 3.0)

    class_weights   = np.array([cw1, cw2, cw3])
    adjusted_probs  = oof_preds * class_weights
    adjusted_probs /= adjusted_probs.sum(axis=1, keepdims=True)
    return accuracy_score(y, np.argmax(adjusted_probs, axis=1))

study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42),
    study_name='class_weight_optimization'
)

In [ ]:
# 500 trials (was 200) for better search coverage
study.optimize(objective, n_trials=500, show_progress_bar=True)

print(f"Best oof acc: {study.best_value:.6f}")
print(f"  class_0 (Low)    = {study.best_params['cw1']:.4f}")
print(f"  class_1 (Medium) = {study.best_params['cw2']:.4f}")
print(f"  class_2 (High)   = {study.best_params['cw3']:.4f}")

best_cw = np.array([study.best_params['cw1'],
                    study.best_params['cw2'],
                    study.best_params['cw3']])

final_oof_probs   = oof * best_cw
final_oof_probs  /= final_oof_probs.sum(axis=1, keepdims=True)

final_test_probs  = pred * best_cw
final_test_probs /= final_test_probs.sum(axis=1, keepdims=True)

final_test_preds  = np.argmax(final_test_probs, axis=1)

In [ ]:
print(f'Overall Balanced Accuracy (after class-weight tuning): {metric(true, final_oof_probs)}')

# Save CSV

In [ ]:
# SAVE OOF TO DISK FOR ENSEMBLES
df_oof  = pd.DataFrame({'xgb': final_oof_probs.flatten()})
df_oof.to_csv(f'{NAME}_oof.csv', index=False)

df_test = pd.DataFrame({'xgb': final_test_probs.flatten()})
df_test.to_csv(f'{NAME}_test.csv', index=False)

print("Saved oof and test predictions to file")

In [ ]:
sub = pd.DataFrame({
    'id': test_ids,
    TARGET: final_test_preds
})
sub[TARGET] = sub[TARGET].map({0: 'Low', 1: 'Medium', 2: 'High'})

sub.to_csv(f'{NAME}_submission.csv', index=False)
print(f"Submission saved: {NAME}_submission.csv")
sub.head(10)